# Fig 6 — Validation integration (circos, associations, enrichment)

Execute cells in order.

Set `DLBCL_DATA_DIR` before the setup cell to override the data root.
Example: `DLBCL_DATA_DIR = '/path/to/data'`

In [1]:
%matplotlib inline

import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dlbcl.notebook_setup import run_notebook_setup

_ctx = run_notebook_setup('validation', 'fig6')
REPO_ROOT = _ctx.repo_root
_paths = _ctx.paths
FIG_DIR = _ctx.fig_dir
adata = _ctx.adata
arch_df = _ctx.arch_df
pred = _ctx.pred
gep = _ctx.gep
surv = _ctx.surv
archetype_label = _ctx.archetype_label
PATIENT_SUBSET = _ctx.patient_subset
OUTDIR = FIG_DIR
OUTDIR.mkdir(parents=True, exist_ok=True)

# Override data root: export DLBCL_DATA_DIR=/path/to/data before running.
print(f"AnnData: {_paths.adata_path}")
print(f"Figures: {FIG_DIR}")

import gc

import pandas as pd
import scanpy as sc

ADATA_PATH = _paths.adata_path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from dlbcl.dlbcl_io import log_wrote, log_saved, write_supplementary_table, rel_path
import dlbcl.validation_figures as vf

plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.family"] = "DejaVu Sans"


FileNotFoundError: AnnData bundle not found at /Users/troynoordenbos/code/anatomy-matters-release/data/DLBCL_location_2026.h5ad.
Download data/DLBCL_location_2026.h5ad — see README.md (Data access).

## Figs 6A–6D — validation integration (location × archetype × LymphoMAP × EcoTyper × COO)
Donut/circos (**4A**), the classifier association dumbbell + pairwise heatmap (**4B**), and enrichment dotplots (**4C** archetype columns, **4D** location columns) using `integration_figures.py`. The dumbbell groups classifiers by whether the archetype or location association (Cramér's V) is higher; filled dots = FDR &lt; 0.05, open dots = not significant. Genomic classifier rings (LymphGen, DLBclass, HMRN, LymphPlex) come from embedded `adata.uns['validation_cohort']['case_classification_validation']`; rings are grey when `genomic_tested` is False.

In [ ]:
from dlbcl.integration_figures import (
    configure_matplotlib,
    compute_group_enrichment_table,
    compute_classifier_pairwise_associations,
    order_patients_hierarchical,
    plot_donut_circos,
    run_association_analysis,
    plot_combined_association_dumbbell,
    plot_classifier_pairwise_heatmap,
    plot_association_enrichment_dotplot_by_classifier,
)

configure_matplotlib()

from dlbcl.validation_classifications import load_case_classification_validation

vc = adata.uns["validation_cohort"]
case_cc = load_case_classification_validation(vc, pred)
meta_int = vf.build_integration_metadata_validation(pred, case_classifications=case_cc)
ordered = order_patients_hierarchical(meta_int)
print(f"Integration cohort: n={len(ordered)} patients")
display(ordered[["Location", "tumorimmune_archetype", "lymphomap", "Lymphoma_Ecotype", "COO_NanoString"]].head())

INT_OUT = FIG_DIR / "integration"
INT_OUT.mkdir(parents=True, exist_ok=True)
ordered.to_csv(INT_OUT / "ordered_patient_metadata.csv")

FIG_6A = FIG_DIR / "fig6A_val_integration_donut.svg"
FIG_6A_LEGEND = FIG_DIR / "fig6A_val_integration_donut_legend.svg"
plot_donut_circos(ordered, FIG_6A, legend_svg=FIG_6A_LEGEND)
log_wrote(FIG_6A, REPO_ROOT)
log_wrote(FIG_6A_LEGEND, REPO_ROOT)

loc_results = run_association_analysis(
    ordered, stratifier="location", out_dir=INT_OUT, pairwise_csv="location_pairwise_results.csv"
)
arch_results = run_association_analysis(
    ordered, stratifier="archetype", out_dir=INT_OUT, pairwise_csv="archetype_pairwise_results.csv"
)
display(loc_results)
display(arch_results)

FIG_6B = FIG_DIR / "fig6B_val_association_dumbbell.svg"
plot_combined_association_dumbbell(loc_results, arch_results, FIG_6B)
log_wrote(FIG_6B, REPO_ROOT)
log_saved(write_supplementary_table(loc_results, REPO_ROOT, "fig6B_val_location_association"), REPO_ROOT)
log_saved(write_supplementary_table(arch_results, REPO_ROOT, "fig6B_val_archetype_association"), REPO_ROOT)

FIG_6B_PAIRWISE = FIG_DIR / "fig6B_val_classifier_pairwise_heatmap.svg"
pairwise_assoc, pairwise_v, pairwise_q = compute_classifier_pairwise_associations(ordered)
pairwise_assoc.to_csv(INT_OUT / "classifier_pairwise_associations.csv", index=False)
log_saved(INT_OUT / "classifier_pairwise_associations.csv", REPO_ROOT)
log_saved(write_supplementary_table(pairwise_assoc, REPO_ROOT, "fig6B_val_classifier_pairwise"), REPO_ROOT)
plot_classifier_pairwise_heatmap(
    pairwise_v,
    pairwise_q,
    FIG_6B_PAIRWISE,
    title="Pairwise classifier association (validation cohort)",
)
log_wrote(FIG_6B_PAIRWISE, REPO_ROOT)

loc_enrichment = compute_group_enrichment_table(ordered, stratifier="location")
arch_enrichment = compute_group_enrichment_table(ordered, stratifier="archetype")
log_saved(write_supplementary_table(loc_enrichment, REPO_ROOT, "fig6D_location_enrichment"), REPO_ROOT)
log_saved(write_supplementary_table(arch_enrichment, REPO_ROOT, "fig6C_archetype_enrichment"), REPO_ROOT)

FIG_6C = FIG_DIR / "fig6C_val_archetype_association_dotplot.svg"
FIG_6D = FIG_DIR / "fig6D_val_location_association_dotplot.svg"
plot_association_enrichment_dotplot_by_classifier(
    ordered,
    stratifier="archetype",
    out_svg=FIG_6C,
    title="Classifier enrichment by predicted immune archetype (validation)",
)
plot_association_enrichment_dotplot_by_classifier(
    ordered,
    stratifier="location",
    out_svg=FIG_6D,
    title="Classifier enrichment by tumor location (validation)",
)
log_wrote(FIG_6C, REPO_ROOT)
log_wrote(FIG_6D, REPO_ROOT)